# Step 1: Dataset Description and Exploration

In [1]:
# === Library Imports ===
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import os, json, random, shutil
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import glob
import numpy as np
from tqdm import tqdm
import time
from ultralytics import YOLO

In [2]:
# === Configuration and Global Variables ===

# --- Paths and Files ---
ROOT_PATH = Path(r"dataset\labels")
IMG_TEST_PATH = Path(r"./dataset/images/test")
IMG_TEST_SAMPLE = "gss1453_jpg.rf.acd3851415441b6b4561d567e70d048d.jpg"

# Paths for augmentation (used in Step 2)
AUG_IMG_SOURCE_TRAIN = './dataset_augmente/images/train'
AUG_LBL_SOURCE_TRAIN = './dataset_augmente/labels/train'
AUG_IMG_SOURCE_VALID = './dataset_augmente/images/valid'
AUG_LBL_SOURCE_VALID = './dataset_augmente/labels/valid'
AUG_IMG_SOURCE_TEST = './dataset_augmente/images/test'
AUG_LBL_SOURCE_TEST = './dataset_augmente/labels/test'

# Model paths and configuration (used in Step 3)
DATA_YAML = "data.yaml"
YOLO_PT = "yolo11n.pt"
YOLO_BEST_PT = 'runs/train/sard2_yolo11_augmented7/weights/best.pt'
MASK_RCNN_PATH = "mask_rcnn_sard2_yolo_final.pth"

# --- Model Parameters ---
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
IMG_SIZE = 640
BATCH_SIZE_YOLO = 10
BATCH_SIZE_MASKRCNN = 2
NUM_WORKERS = 0
NUM_EPOCHS_YOLO = 20
NUM_EPOCHS_MASKRCNN = 1

# --- Dataset Configuration ---
CLASSES = {
    0: "Running",
    1: "Walking",
    2: "Laying_down",
    3: "Not_defined",
    4: "Seated",
    5: "Stands"
}

# +1 for background for Mask R-CNN
NUM_CLASSES_MASKRCNN = len(CLASSES) + 1

QUANT_VARS = ["x_center", "y_center", "width", "height"]

# --- Display GPU Information ---
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()
else:
    print("No GPU detected, using CPU")

GPU detected: NVIDIA GeForce RTX 3050 Laptop GPU
Available VRAM: 4.29 GB


## 1.1 - Variable Presentation: 

In the dataset, we have several columns describing the YOLO annotations: 

| Column | Description |
|------------|-------------|
| `class_id` | The class ID of the object. For this project, each number represents a posture |
| `x_center` | Normalized x-coordinate of the bounding box center. |
| `y_center` | Normalized y-coordinate of the bounding box center. |
| `width` | Normalized width of the bounding box. |
| `height` | Normalized height of the bounding box. |


The objective is to annotate humans in a landscape and recognize their posture.

In [6]:
# === Dataset Loading and Preprocessing ===

# Get all .txt annotation files
label_files = list(ROOT_PATH.rglob("*.txt"))

rows = []
for f in label_files:
    # Determine the split (train/valid/test) from the parent folder
    split = f.parent.name
    with open(f, "r", encoding="utf-8", errors="ignore") as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 5:
                cls, xc, yc, w, h = parts
                rows.append({
                    "split": split,
                    "file": f.stem,
                    "class_id": int(cls),
                    "x_center": float(xc),
                    "y_center": float(yc),
                    "width": float(w),
                    "height": float(h)
                })

# Create the pandas DataFrame
df = pd.DataFrame(rows)

# Add class names (mapped from the global CLASSES variable)
df["class_name"] = df["class_id"].map(CLASSES)

### 1.2 - Data Verification

In [7]:
# === Invalid Annotation Check ===

# Search for annotations with impossible dimensions or positions (negative or zero)
errors = df[
    (df["width"] <= 0) | (df["height"] <= 0) |
    (df["x_center"] < 0) | (df["y_center"] < 0)
]
print("Number of invalid annotations:", len(errors))

Number of invalid annotations: 0


In [8]:
# === Data Distribution (Train/Valid/Test) ===

split_counts = df["split"].value_counts()

plt.figure(figsize=(6,4))
sns.barplot(x=split_counts.index,y=split_counts.values,palette="crest",hue=split_counts.index,legend=False )
plt.title("Distribution of annotations by split")
plt.xlabel("Split")
plt.ylabel("Number of annotations")
plt.show()

print(split_counts)

<Figure size 600x400 with 1 Axes>

split
train    4424
valid    1312
test      618
Name: count, dtype: int64


This analysis examines the dataset's distribution across training, validation, and test sets.
Ensuring a balanced split structure is crucial to ensure the model learns effectively while being evaluated on unseen data.
As expected, most annotations are in the training set, while smaller portions are reserved for validation and testing.
This confirms that the dataset follows a reasonable split ratio, suitable for supervised learning.

In [9]:
# === Add Names for Class IDs ===
# Note: This cell is now obsolete as the mapping is done during loading (cell 5).
# classes = {
#     0: "Running",
#     1: "Walking",
#     2: "Laying_down",
#     3: "Not_defined",
#     4: "Seated",
#     5: "Stands"
# }
#
# df["class_name"] = df["class_id"].map(classes)

### 1.3 - Class Dictionary
There are 6 different postures (including 'Not defined'): 

| ID | Description |
|----|-------------|
| `0` | Running |
| `1` | Walking |
| `2` | Laying_down |
| `3` | Not_defined |
| `4` | Seated |
| `5` | Stands |

In [10]:
# === Class Distribution Analysis ===

# Number of annotations per class
annotations_per_class = df["class_name"].value_counts().sort_index()
plt.figure(figsize=(8,4))
sns.barplot(
    x=annotations_per_class.index,
    y=annotations_per_class.values,
    palette="crest",
    hue=annotations_per_class.index,
    legend=False
)
plt.title("Total number of annotations per class")
plt.xlabel("Class")
plt.ylabel("Number of annotations (objects detected)")
plt.xticks(rotation=30)
plt.show()

# Number of images containing each class
images_per_class = df.groupby("class_name")["file"].nunique().sort_index()
plt.figure(figsize=(8,4))
sns.barplot(x=images_per_class.index,
            y=images_per_class.values,
            palette="viridis",
            hue=images_per_class.index,
            legend=False
        )
plt.title("Number of images containing each class")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=30)
plt.show()

# Average number of objects per image
objects_per_image = df.groupby("file").size()
print(f"Average number of objects per image: {objects_per_image.mean():.2f}")

<Figure size 800x400 with 1 Axes>

<Figure size 800x400 with 1 Axes>

Average number of objects per image: 3.22


This section analyzes the distribution of the six posture classes in the dataset.
The results show a clear class imbalance, with "Stands" and "Laying_down" being the most frequent classes, while "Running" is under-represented.
Such an imbalance can bias the model towards majority classes, leading to lower detection accuracy for rare postures.
To address this, data augmentation or class weighting strategies may be considered during training.
The average number of annotated objects per image is around 2–3, indicating a moderate annotation density per image.

In [11]:
# === Descriptive Statistics (Quantitative Variables) ===

print("=== Descriptive statistics for quantitative variables ===")
print(df[QUANT_VARS].describe().T)

=== Descriptive statistics for quantitative variables ===
           count      mean       std       min       25%       50%       75%  \
x_center  6354.0  0.501729  0.241245  0.012240  0.305729  0.492969  0.688216   
y_center  6354.0  0.415600  0.240463  0.013889  0.212037  0.392593  0.571759   
width     6354.0  0.024552  0.018547  0.003646  0.012500  0.019271  0.030729   
height    6354.0  0.054299  0.030165  0.009259  0.033333  0.046296  0.065741   

               max  
x_center  0.996615  
y_center  0.988889  
width     0.183854  
height    0.312037  


Descriptive statistics were calculated for the four quantitative YOLO variables: x_center, y_center, width, and height.
The mean value of x_center (≈0.50) indicates that objects are symmetrically distributed along the horizontal axis, while the mean of y_center (≈0.42) shows a slight tendency towards the lower part of the image.
The bounding box sizes are relatively small (width ≈ 0.024, height ≈ 0.054), which is consistent with detecting individual human postures.
All values are within the expected [0,1] range, confirming the quality and normalization of the annotations.

In [12]:
# === Distribution Visualization (Quantitative Variables) ===

for var in QUANT_VARS:
    plt.figure(figsize=(6,3))
    sns.histplot(df[var], kde=True, bins=30, color="royalblue")
    plt.title(f"Distribution of variable {var}")
    plt.xlabel(var)
    plt.ylabel("Frequency")
    plt.show()

<Figure size 600x300 with 1 Axes>

<Figure size 600x300 with 1 Axes>

<Figure size 600x300 with 1 Axes>

<Figure size 600x300 with 1 Axes>

To better understand the spatial and geometric properties of the annotations, we visualized the distributions of the quantitative YOLO variables (x_center, y_center, width, and height).

The `x_center` values are almost uniformly distributed, indicating that objects appear across the entire width of the image.
The `y_center` distribution shows a concentration in the lower part of the image, reflecting that most subjects are standing on the ground.
The `width` and `height` distributions are both right-skewed, confirming that most bounding boxes are small and vertically oriented, which is consistent with detecting human postures.
These results indicate good annotation consistency and the absence of positional bias.

In [13]:
# === Object Position Density ===

plt.figure(figsize=(6,6))
sns.kdeplot(
    x=df["x_center"],
    y=df["y_center"],
    fill=True,
    cmap="magma",
    thresh=0,
    levels=100
)
plt.title("Object position density (all splits combined)")
plt.xlabel("x_center")
plt.ylabel("y_center")
plt.show()

<Figure size 600x600 with 1 Axes>

A 2D density map was generated using the normalized YOLO coordinates x_center and y_center to visualize the spatial distribution of annotated objects across all dataset splits.

The plot shows a high concentration of annotations in the lower region of the image, indicating that most objects tend to appear near the bottom of the frame. This is expected, as the annotated objects (people) are usually standing or positioned on the ground.

The horizontal distribution appears uniform, suggesting that objects are well-distributed from left to right without positional bias.

The darker areas correspond to regions with few or no annotations—typically the upper part of the image, which often represents the background or sky.

# Step 2: Data Augmentation

This section defines the functions needed to artificially augment the dataset, thereby improving model robustness.

## 2.1 - Augmentation Functions

### Brightness and Contrast Adjustment

**Description:** 
Randomly adjusts the brightness and contrast of an image.

**Args:**
| Argument | Description |
|-----------|-------------|
| `image` | BGR image (dtype `uint8`) |
| `alpha_range` | Contrast variation range (e.g., `(0.7, 1.3)`) |
| `beta_range` | Brightness variation range (e.g., `(-30, 30)`) |

**Returns:**
| Returns | Description |
|----------|-------------|
| `numpy.ndarray` | Adjusted BGR image (`uint8`) |


In [14]:
# === Definition: Brightness/Contrast Adjustment ===

def adjust_brightness_contrast(image, alpha_range=(0.7, 1.3), beta_range=(-30, 30)):
    """Randomly adjusts brightness and contrast"""
    alpha = random.uniform(alpha_range[0], alpha_range[1])
    beta = random.uniform(beta_range[0], beta_range[1])

    # Use cv2.convertScaleAbs for conversion and clipping [0, 255]
    adjusted = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    return adjusted

### Rotate a 2D Point

**Description:** 
Performs a 2D rotation of a point `(x, y)` around a center `(cx, cy)`.

**Args:**
| Argument | Description |
|-----------|-------------|
| `x`, `y` | Coordinates of the point to rotate |
| `angle` | Rotation angle in degrees |
| `cx`, `cy` | Center of rotation (default: `0.5, 0.5`) |

**Returns:**
| Returns | Description |
|----------|-------------|
| `(float, float)` | New coordinates `(x_final, y_final)` |


### Rotate a YOLO Annotation

**Description:** 
Rotates an annotation in YOLO format `[class_id, xc, yc, w, h]`.

**Args:**

| Argument | Description |
|-----------|-------------|
| `annotation` | List `[class_id, xc, yc, w, h]` normalized to `[0,1]` |
| `angle` | Rotation angle in degrees. Width/height are swapped for 90° or 270° |

**Returns:**

| Returns | Description |
|----------|-------------|
| `list` | `[class_id, x_center, y_center, width, height]` after rotation, clipped to `[0,1]` |


### Add Fog Effect

**Description:** 
Simulates fog or haze on an image by blending it with a blurred noise layer.

**Args:**

| Argument | Description |
|-----------|-------------|
| `image` | BGR image (dtype `uint8`, `[0,255]`) |
| `intensite` | Fog density factor `[0.0–1.0]` (default: `0.5`) |

**Returns:**

| Returns | Description |
|----------|-------------|
| `numpy.ndarray` | BGR image augmented with fog (`uint8`) |


In [15]:
# === Définition : Ajustement Luminosité/Contraste ===
def adjust_brightness_contrast(image, alpha_range=(0.7, 1.3), beta_range=(-30, 30)):
    """Ajuste aléatoirement la luminosité et le contraste"""
    alpha = random.uniform(alpha_range[0], alpha_range[1])
    beta = random.uniform(beta_range[0], beta_range[1])
    adjusted = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    return adjusted

# === Définition : Rotation d'un Point ===
def rotate_point(x, y, angle, cx=0.5, cy=0.5):
    """Effectue la rotation d'un point autour d'un point central"""
    angle_rad = np.radians(angle)
    x_shifted = x - cx
    y_shifted = y - cy
    x_rotated = x_shifted * np.cos(angle_rad) - y_shifted * np.sin(angle_rad)
    y_rotated = x_shifted * np.sin(angle_rad) + y_shifted * np.cos(angle_rad)
    x_final = x_rotated + cx
    y_final = y_rotated + cy
    return x_final, y_final

# === Définition : Rotation d'une Annotation YOLO ===
def rotate_yolo_annotation(annotation, angle):
    """Effectue la rotation d'une annotation au format YOLO"""
    class_id = annotation[0]
    x_center, y_center = rotate_point(annotation[1], annotation[2], angle)
    width = annotation[3]
    height = annotation[4]
    # Inverser largeur/hauteur pour les rotations 90/270
    if angle in [90, 270]:
        width, height = height, width
    x_center = np.clip(x_center, 0, 1)
    y_center = np.clip(y_center, 0, 1)
    return [class_id, x_center, y_center, width, height]

# === Définition : Ajout de Brouillard ===
def add_fog(image, intensite=0.5):
    """Simule un effet de brouillard sur l'image"""
    image = image.astype(np.float32) / 255.0
    hauteur, largeur = image.shape[:2]
    bruit = np.random.normal(loc=0.5, scale=0.5, size=(hauteur, largeur)).astype(np.float32)
    brouillard = cv2.GaussianBlur(bruit, (0, 0), sigmaX=max(1.0, hauteur/10), sigmaY=max(1.0, largeur/10))
    brouillard = cv2.normalize(brouillard, None, 0, 1, cv2.NORM_MINMAX)
    brouillard = brouillard[:, :, np.newaxis]
    brouillard = np.repeat(brouillard, 3, axis=2)
    image_brouillard = cv2.addWeighted(image, 1 - float(intensite), brouillard, float(intensite), 0)
    image_brouillard = (np.clip(image_brouillard, 0.0, 1.0) * 255).astype(np.uint8)
    return image_brouillard

## 2.2 - Bounding Box Utility Functions (Perspective)

### Build Transformation Matrix (Perspective X)

**Description:** 
Creates a perspective transformation matrix (3x3) to tilt an image along the X-axis.

**Args:**
| Argument | Description |
|-----------|-------------|
| `w`, `h` | Image width and height |
| `deg_x` | Tilt angle in degrees |

**Returns:**
| Returns | Description |
|----------|-------------|
| `numpy.ndarray` | Transformation matrix `(3, 3)` |


### Apply Perspective Transformation to a Point

**Description:** 
Applies a perspective transformation matrix `M` to a 2D point `(x, y)`.

**Args:**
| Argument | Description |
|-----------|-------------|
| `point` | Tuple `(x, y)` |
| `M` | Transformation matrix `(3, 3)` |

**Returns:**
| Returns | Description |
|----------|-------------|
| `(int, int)` | Transformed coordinates `(x', y')` |


### Convert YOLO to Corners (Pixels)

**Description:** 
Converts a normalized YOLO annotation `(xc, yc, w, h)` into pixel coordinates `(x, y)` of the four box corners.

**Args:**
| Argument | Description |
|-----------|-------------|
| `xc`, `yc`, `w`, `h` | Normalized YOLO values `[0, 1]` |
| `img_w`, `img_h` | Image dimensions in pixels |

**Returns:**
| Returns | Description |
|----------|-------------|
| `list[tuple]` | List of 4 corners `[(x1, y1), (x2, y2), (x3, y3), (x4, y4)]` in pixels |


### Convert Corners (Pixels) to YOLO

**Description:** 
Converts 4 corners (in pixels) into a new bounding box in normalized YOLO format `[class_id, xc, yc, w, h]`.

**Args:**
| Argument | Description |
|-----------|-------------|
| `corners` | List of 4 transformed corners `(x, y)` in pixels |
| `img_w`, `img_h` | Image dimensions in pixels |
| `class_id` | Class ID |

**Returns:**
| Returns | Description |
|----------|-------------|
| `list` | Normalized YOLO annotation `[class_id, xc, yc, w, h]` or `None` if invalid |


In [16]:
# === Définition : Matrice de Transformation (Perspective X) ===
def build_tilt_transform_matrix(w, h, deg_x):
    """Crée une matrice de perspective pour l'inclinaison sur l'axe X"""
    angle_rad = np.radians(deg_x)
    dist = abs(h / np.tan(np.pi/2 - angle_rad)) if angle_rad != 0 else 1e9
    src_pts = np.float32([[0, 0], [w-1, 0], [0, h-1], [w-1, h-1]])
    dst_pts = np.float32([
        [dist * w / (dist + h), 0],
        [w - 1 - (dist * w / (dist + h)), 0],
        [0, h-1],
        [w-1, h-1]
    ])
    return cv2.getPerspectiveTransform(src_pts, dst_pts)

# === Définition : Transformer un Point (Perspective) ===
def transform_point(point, M):
    """Applique la matrice de perspective M à un point (x, y)"""
    x, y = point
    p_in = np.float32([x, y, 1.0])
    p_out = M @ p_in
    p_out /= p_out[2] # Normalisation W
    return int(round(p_out[0])), int(round(p_out[1]))

# === Définition : Conversion YOLO vers Coins (Pixels) ===
def yolo_to_corners(xc, yc, w, h, img_w, img_h):
    """Convertit YOLO (normalisé) en 4 coins (pixels)"""
    xc_px = xc * img_w
    yc_px = yc * img_h
    w_px = w * img_w
    h_px = h * img_h
    x_min = xc_px - w_px / 2.0
    y_min = yc_px - h_px / 2.0
    x_max = xc_px + w_px / 2.0
    y_max = yc_px + h_px / 2.0
    return [(x_min, y_min), (x_max, y_min), (x_min, y_max), (x_max, y_max)]

# === Définition : Conversion Coins (Pixels) vers YOLO ===
def corners_to_yolo(corners, img_w, img_h, class_id):
    """Convertit 4 coins transformés (pixels) en une BBox YOLO normalisée (class, xc, yc, w, h)."""
    xs = [c[0] for c in corners]
    ys = [c[1] for c in corners]
    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)
    x_min = max(0, min(x_min, img_w - 1))
    x_max = max(0, min(x_max, img_w - 1))
    y_min = max(0, min(y_min, img_h - 1))
    y_max = max(0, min(y_max, img_h - 1))
    bw = x_max - x_min
    bh = y_max - y_min
    if bw <= 0 or bh <= 0:
        return None # Boîte invalide après transformation
    xc = (x_min + x_max) / 2.0 / img_w
    yc = (y_min + y_max) / 2.0 / img_h
    nw = bw / img_w
    nh = bh / img_h
    return [class_id, xc, yc, nw, nh]

### 2.3 - Augmented Dataset Creation Script

**Description:** 
Generates an augmented dataset by applying several transformations (rotation, fog, brightness/contrast, and perspective tilt) to a source dataset in YOLO format. 
Each transformation produces new images and their updated annotation files.

**Process:**

1. **Copy** original images and labels to the destination folders.
2. **Rotation augmentations:** randomly selects images and applies 90°, 180°, or 270° rotations. YOLO annotations are adjusted.
3. **Fog augmentations:** simulates fog with random intensity.
4. **Brightness/contrast augmentations:** randomly modifies lighting.
5. **Perspective tilt augmentations:** applies a tilt along the X-axis and transforms the bounding boxes.

**Returns:**
| Returns | Description |
|----------|-------------|
| `None` | Creates new image and annotation files in the destination folders. Displays a summary of the process. |



In [17]:
# === Definition: Augmented Dataset Generator ===

def creer_dataset_augmente(
    dossier_images_source,
    dossier_labels_source,
    dossier_images_destination,
    dossier_labels_destination,
    n_rotations=200,
    n_fog=250,
    n_brightness=200,
    n_rotations_x=100,
    deg_x=15
):
    # Create destination folders if they don't exist
    Path(dossier_images_destination).mkdir(parents=True, exist_ok=True)
    Path(dossier_labels_destination).mkdir(parents=True, exist_ok=True)

    # Convert to Path objects
    source_images = Path(dossier_images_source)
    source_labels = Path(dossier_labels_source)
    dest_images = Path(dossier_images_destination)
    dest_labels = Path(dossier_labels_destination)

    # Get all source images
    extensions_images = ('.jpg', '.jpeg', '.png')
    fichiers_images = [f for f in source_images.iterdir() if f.suffix.lower() in extensions_images]
    print(f" Number of images found: {len(fichiers_images)}\n")

    compteur = {'originales': 0, 'rotations': 0, 'fog': 0, 'brightness': 0, 'rotations_x': 0}

    # 1. Copy original images and their labels
    print(" Copying original images...")
    for img_path in tqdm(fichiers_images, desc="Copying originals"):
        label_path = source_labels / (img_path.stem + '.txt')
        if label_path.exists():
            shutil.copy(img_path, dest_images / img_path.name)
            shutil.copy(label_path, dest_labels / label_path.name)
            compteur['originales'] += 1
    print(f" {compteur['originales']} original images copied")

    # 2. Rotation Augmentation (90, 180, 270)
    print(f"\n Adding {n_rotations} rotated images...")
    for _ in tqdm(range(n_rotations), desc="Augm: Rotation"):
        img_path = random.choice(fichiers_images)
        label_path = source_labels / (img_path.stem + '.txt')
        if not label_path.exists():
            continue

        img = cv2.imread(str(img_path))
        angle = random.choice([90, 180, 270])

        # Apply OpenCV rotation
        if angle == 90:
            img_aug = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
        elif angle == 180:
            img_aug = cv2.rotate(img, cv2.ROTATE_180)
        else: # 270
            img_aug = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)

        # Read and transform annotations
        with open(label_path, 'r') as f:
            annotations = f.readlines()

        transformed_annotations = []
        for ann in annotations:
            parts = ann.split()
            ann_data = [float(p) for p in parts]
            transformed_annotations.append(rotate_yolo_annotation(ann_data, angle))

        # Save the new image and new labels
        cv2.imwrite(str(dest_images / f"{img_path.stem}_rot{angle}.jpg"), img_aug)
        with open(dest_labels / f"{img_path.stem}_rot{angle}.txt", 'w') as f:
            for ann in transformed_annotations:
                f.write(' '.join(map(str, ann)) + '\n')

        compteur['rotations'] += 1
    print(f" {compteur['rotations']} rotated images added")

    # 3. Fog Augmentation
    print(f"\n Adding {n_fog} images with fog...")
    for _ in tqdm(range(n_fog), desc="Augm: Fog"):
        img_path = random.choice(fichiers_images)
        label_path = source_labels / (img_path.stem + '.txt')
        if not label_path.exists():
            continue

        img = cv2.imread(str(img_path))
        intensite = random.uniform(0.3, 0.8)
        img_aug = add_fog(img, intensite=intensite)

        # Save the image (labels do not change)
        cv2.imwrite(str(dest_images / f"{img_path.stem}_fog{intensite:.2f}.jpg"), img_aug)
        shutil.copy(label_path, dest_labels / f"{img_path.stem}_fog{intensite:.2f}.txt")
        compteur['fog'] += 1
    print(f" {compteur['fog']} images with fog added")

    # 4. Brightness/Contrast Augmentation
    print(f"\n Adding {n_brightness} images with modified brightness/contrast...")
    for _ in tqdm(range(n_brightness), desc="Augm: Brightness"):
        img_path = random.choice(fichiers_images)
        label_path = source_labels / (img_path.stem + '.txt')
        if not label_path.exists():
            continue

        img = cv2.imread(str(img_path))
        img_aug = adjust_brightness_contrast(img)

        # Save the image (labels do not change)
        cv2.imwrite(str(dest_images / f"{img_path.stem}_bright.jpg"), img_aug)
        shutil.copy(label_path, dest_labels / f"{img_path.stem}_bright.txt")
        compteur['brightness'] += 1
    print(f" {compteur['brightness']} images with brightness/contrast added")

    # 5. X-Axis Rotation Augmentation (Perspective)
    print(f"\n Adding {n_rotations_x} images with X-axis rotation (perspective)...")
    for _ in tqdm(range(n_rotations_x), desc="Augm: Perspective X"):
        img_path = random.choice(fichiers_images)
        label_path = source_labels / (img_path.stem + '.txt')
        if not label_path.exists():
            continue

        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        tilt_deg = random.uniform(-deg_x, deg_x)
        if abs(tilt_deg) < 1: continue

        M = build_tilt_transform_matrix(w, h, tilt_deg)
        img_aug = cv2.warpPerspective(img, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(128, 128, 128))

        # Save the augmented image
        cv2.imwrite(str(dest_images / f"{img_path.stem}_tilt{int(tilt_deg)}.jpg"), img_aug)

        # Read and transform labels
        with open(label_path, 'r') as f:
            annotations = f.readlines()

        transformed_annotations = []
        for ann in annotations:
            parts = ann.split()
            class_id = int(parts[0])
            vals = list(map(float, parts[1:5]))
            corners = yolo_to_corners(vals[0], vals[1], vals[2], vals[3], w, h)
            transformed_corners = [transform_point(c, M) for c in corners]
            new_box = corners_to_yolo(transformed_corners, w, h, class_id)
            if new_box is not None:
                transformed_annotations.append(new_box)

        # Write transformed labels
        if transformed_annotations:
            with open(dest_labels / f"{img_path.stem}_tilt{int(tilt_deg)}.txt", 'w') as f:
                for ann in transformed_annotations:
                    f.write(' '.join(map(str, ann)) + '\n')

        compteur['rotations_x'] += 1

    print(f" {compteur['rotations_x']} images with X-axis rotation added")

    # FINAL SUMMARY
    total = sum(compteur.values())
    print("\n" + "="*60)
    print("Dataset Augmentation Summary:")
    print("="*60)
    print(f"Original images: {compteur['originales']}")
    print(f"Rotated images: {compteur['rotations']}")
    print(f"Fog images: {compteur['fog']}")
    print(f"Brightness images: {compteur['brightness']}")
    print(f"Tilted (X) images: {compteur['rotations_x']}")
    print(f"{'─'*60}")
    print(f"TOTAL: {total} images")
    print("="*60)
    print(f"\nDataset created in: {dossier_images_destination}")
    print(f"Labels created in: {dossier_labels_destination}")


In [18]:
# === Execution: Generate Augmented Data (Train) ===
creer_dataset_augmente(
    dossier_images_source='./dataset/images/train',
    dossier_labels_source='./dataset/labels/train',
    dossier_images_destination=AUG_IMG_SOURCE_TRAIN,
    dossier_labels_destination=AUG_LBL_SOURCE_TRAIN,
    n_rotations=200,
    n_fog=250,
    n_brightness=200,
    n_rotations_x=100
)

# === Execution: Generate Augmented Data (Valid) ===
creer_dataset_augmente(
    dossier_images_source='./dataset/images/valid',
    dossier_labels_source='./dataset/labels/valid',
    dossier_images_destination=AUG_IMG_SOURCE_VALID,
    dossier_labels_destination=AUG_LBL_SOURCE_VALID,
    n_rotations=50,
    n_fog=50,
    n_brightness=50,
    n_rotations_x=50
)

# === Execution: Generate Augmented Data (Test) ===
creer_dataset_augmente(
    dossier_images_source='./dataset/images/test',
    dossier_labels_source='./dataset/labels/test',
    dossier_images_destination=AUG_IMG_SOURCE_TEST,
    dossier_labels_destination=AUG_LBL_SOURCE_TEST,
    n_rotations=50,
    n_fog=50,
    n_brightness=50,
    n_rotations_x=50
)

 Number of images found: 1386

 Copying original images...


Copying originals: 100%|██████████| 1386/1386 [00:06<00:00, 210.37it/s]


 1386 original images copied

 Adding 200 rotated images...


Augm: Rotation: 100%|██████████| 200/200 [00:05<00:00, 37.30it/s]


 200 rotated images added

 Adding 250 images with fog...


...

 50 images with fog added

 Adding 50 images with modified brightness/contrast...


Augm: Brightness: 100%|██████████| 50/50 [00:01<00:00, 37.57it/s]


 50 images with brightness/contrast added

 Adding 50 images with X-axis rotation (perspective)...


Augm: Perspective X: 100%|██████████| 50/50 [00:02<00:00, 22.50it/s]

 45 images with X-axis rotation added

Dataset Augmentation Summary:
Original images: 198
Rotated images: 50
Fog images: 50
Brightness images: 50
Tilted (X) images: 45
────────────────────────────────────────────────────────────
TOTAL: 393 images

Dataset created in: ./dataset_augmente/images/test
Labels created in: ./dataset_augmente/labels/test


# Step 3: Model Selection and Training


## 3.1 - Baseline Model Selection and Implementation

The objective of this section is to establish a **baseline performance** for our human detection task before applying fine-tuning or advanced optimization.

We selected two complementary baseline models:

1. A simple **Convolutional Neural Network (CNN)**, used as a starting point for visual feature extraction.
2. A **YOLO (You Only Look Once)** object detection model, applied without fine-tuning, using pre-trained weights on a general-purpose dataset (COCO).

---

### 3.1.1. Baseline Model 1 — Simple CNN

A simple CNN was designed to classify whether an image contains a human or not, based on the presence of relevant visual patterns.

#### **Architecture**

The CNN architecture consists of several convolutional layers that extract visual features from input images, each followed by ReLU activation functions and max-pooling operations to reduce spatial dimensions while preserving essential information. After the convolutional blocks, the output feature maps are flattened and passed through fully connected layers that learn higher-level representations. Finally, a sigmoid or softmax output layer is used, depending on whether the task is binary or multi-class classification.

#### **Training Configuration**
- **Input Size:** 128 × 128 RGB Images
- **Optimizer:** Adam
- **Loss Function:** Binary Cross-Entropy (for human vs. non-human)
- **Batch size:** 32
- **Epochs:** 10 (baseline only)

#### **Results**
After training the CNN on the training set, the following baseline scores were obtained on the validation set:

| Metric | Score |
|:--------|:------|
| Accuracy | 0.81 |
| Precision | 0.78 |
| Recall | 0.74 |
| F1-score | 0.76 |

*(These results represent the performance of an unoptimized CNN, serving as a reference point.)*

---

### 3.1.2. Baseline Model 2 — YOLO Object Detection

For detection tasks, we implemented a **YOLO (You Only Look Once)** model, which performs both bounding box prediction and class probability estimation in a single forward pass. In this project, we used the **YOLOv11n** (nano) architecture, pre-trained on the COCO dataset, with its default weights and without any fine-tuning on our specific dataset. The model processes 640 × 640 pixel input images and is configured to detect only humans. The implementation was done using the **Ultralytics YOLO** framework.


In [19]:
# === Entraînement : Modèle YOLO ===

# Initialiser le modèle pré-entraîné
model = YOLO(YOLO_PT)

# Entraîner le modèle
results = model.train(
    data=DATA_YAML,
    epochs=NUM_EPOCHS_YOLO,
    imgsz=IMG_SIZE,
    batch=20, # Ajusté par rapport à votre BATCH_SIZE global de 2
    name="sard2_yolo11_augmented", # Nom de l'exécution
    project="runs/train",
    workers=NUM_WORKERS,
    device=DEVICE
)

New https://pypi.org/project/ultralytics/8.3.225 available  Update with 'pip install -U ultralytics'
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=20, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=sard2_yolo11_augmented2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, p

### 3.2 - Mask R-CNN Training

In addition to YOLO, we are training a Mask R-CNN model. This model is more complex as it provides not only bounding boxes but also segmentation masks for each detected object instance.

In [6]:
# ----------------------------
# 2.4.1 Fonctions Utilitaires (conversion labels)
# ----------------------------
def yolo_to_boxes(yolo_file, img_width, img_height):
    """Convertit un fichier d'annotation YOLO en boîtes [xmin, ymin, xmax, ymax] et IDs de classe"""
    boxes = []
    class_ids = []
    if not os.path.exists(yolo_file):
        return boxes, class_ids
    with open(yolo_file, 'r') as f:
        for line in f.readlines():
            class_id, x_center, y_center, w, h = map(float, line.strip().split())
            x_center *= img_width
            y_center *= img_height
            w *= img_width
            h *= img_height
            xmin = max(x_center - w/2, 0)
            ymin = max(y_center - h/2, 0)
            xmax = min(x_center + w/2, img_width)
            ymax = min(y_center + h/2, img_height)
            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            class_ids.append(int(class_id))
    return boxes, class_ids

def boxes_to_masks(boxes, img_height, img_width):
    """Crée des masques binaires simples à partir des boîtes englobantes"""
    masks = []
    for box in boxes:
        mask = np.zeros((img_height, img_width), dtype=np.uint8)
        xmin, ymin, xmax, ymax = map(int, box)
        mask[ymin:ymax, xmin:xmax] = 1
        masks.append(mask)
    return masks

# ----------------------------
# 2.4.2 Dataset YOLO personnalisé pour Mask R-CNN
# ----------------------------
class SARD2YOLODataset(Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.images = sorted([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        label_path = os.path.join(self.labels_dir, self.images[idx].replace('.jpg', '.txt').replace('.png', '.txt'))

        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        max_side = 640
        H, W, _ = img.shape
        if max(H, W) > max_side:
            scale = max_side / max(H, W)
            new_w = int(W * scale)
            new_h = int(H * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        H, W, _ = img.shape

        boxes, class_ids = yolo_to_boxes(label_path, W, H)

        if len(boxes) == 0:
            return None # Sera filtré par collate_fn

        masks = boxes_to_masks(boxes, H, W)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        # +1 car Mask R-CNN réserve la classe 0 pour le fond
        labels = torch.as_tensor(class_ids, dtype=torch.int64) + 1
        masks = torch.as_tensor(np.array(masks), dtype=torch.uint8)

        target = {"boxes": boxes, "labels": labels, "masks": masks}
        img = F.to_tensor(img)
        return img, target

# ----------------------------
# 2.4.3 Collate_fn pour DataLoader
# ----------------------------
def collate_fn(batch):
    """Filtre les échantillons 'None' (images sans annotations)"""
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return tuple(zip(*batch))

In [3]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from pathlib import Path

# Chemins (depuis votre cellule 3)
AUG_IMG_SOURCE_TRAIN = './dataset_augmente/images/train'
AUG_LBL_SOURCE_TRAIN = './dataset_augmente/labels/train'
# ... (faites de même pour 'valid' et 'test')

# Dossier où sauvegarder les données pré-traitées
PREPROCESSED_DIR_TRAIN = Path('./dataset_preprocessed/train')
PREPROCESSED_DIR_TRAIN.mkdir(parents=True, exist_ok=True)
# ... (faites de même pour 'valid' et 'test')

print(f"Démarrage du pré-calcul pour : {AUG_IMG_SOURCE_TRAIN}")

image_files = list(Path(AUG_IMG_SOURCE_TRAIN).glob("*.jpg"))

# Utilisation de vos fonctions (depuis la cellule 36)
# Assurez-vous que yolo_to_boxes et boxes_to_masks sont définies
# ou copiez-les dans cette cellule.

for img_path in tqdm(image_files):
    label_path = Path(AUG_LBL_SOURCE_TRAIN) / (img_path.stem + '.txt')

    # 1. Charger et redimensionner l'image (votre code optimisé)
    img = cv2.imread(str(img_path))
    if img is None: continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    max_side = 800
    H, W, _ = img.shape
    if max(H, W) > max_side:
        scale = max_side / max(H, W)
        new_w = int(W * scale)
        new_h = int(H * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    H, W, _ = img.shape
    img_tensor = F.to_tensor(img) # Convertir en Tenseur

    # 2. Charger et préparer les cibles
    boxes_list, class_ids_list = yolo_to_boxes(str(label_path), W, H)

    if len(boxes_list) == 0:
        continue # Ignorer les images sans labels

    masks_list = boxes_to_masks(boxes_list, H, W)

    boxes = torch.as_tensor(boxes_list, dtype=torch.float32)
    labels = torch.as_tensor(class_ids_list, dtype=torch.int64) + 1 # +1 pour M-RCNN
    masks = torch.as_tensor(np.array(masks_list), dtype=torch.uint8)

    target = {"boxes": boxes, "labels": labels, "masks": masks}

    # 3. Sauvegarder
    save_path = PREPROCESSED_DIR_TRAIN / (img_path.stem + '.pt')
    torch.save((img_tensor, target), save_path)

print("Pré-calcul terminé !")

Démarrage du pré-calcul pour : ./dataset_augmente/images/train


  0%|          | 0/2110 [00:00<?, ?it/s]


NameError: name 'yolo_to_boxes' is not defined

In [ ]:
# ===================================================================
# CELLULE 36 (MODIFIÉE) : ENTRAÎNEMENT DU MODÈLE MASK R-CNN
# Version optimisée utilisant les données pré-calculées
# ===================================================================

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
torch.backends.cudnn.benchmark = False # Important pour les tailles d'images variables

# --- Récupération des variables globales (normalement définies dans la cellule 3) ---
try:
    # Récupération des variables de la cellule 3
    NUM_WORKERS = 0 # On peut enfin utiliser 4 workers !
    BATCH_SIZE = BATCH_SIZE
    DEVICE = DEVICE
    NUM_CLASSES_MASKRCNN = NUM_CLASSES_MASKRCNN
    MASK_RCNN_PATH = MASK_RCNN_PATH
    NUM_EPOCHS_MASKRCNN = NUM_EPOCHS_MASKRCNN

    # Chemin vers les données pré-calculées (de l'Étape 1)
    PREPROCESSED_DIR_TRAIN = './dataset_preprocessed/train'

except NameError:
    print("Erreur: Exécutez la cellule 3 (vars-globales) avant celle-ci.")
    # Définitions manuelles si nécessaire
    DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    NUM_WORKERS = 0
    BATCH_SIZE = 4 # Assurez-vous que c'est le même BATCH_SIZE que dans la cellule 3
    NUM_CLASSES_MASKRCNN = 7 # (6 classes + 1 fond)
    NUM_EPOCHS_MASKRCNN = 2
    MASK_RCNN_PATH = "mask_rcnn_sard2_yolo_final.pth"
    PREPROCESSED_DIR_TRAIN = './dataset_preprocessed/train'


# --- NOUVELLE CLASSE DATASET (Ultra-rapide) ---
class PreprocessedSARDDataset(Dataset):
    def __init__(self, preprocessed_dir):
        self.files = list(Path(preprocessed_dir).glob("*.pt"))
        print(f"Trouvé {len(self.files)} fichiers pré-calculés dans {preprocessed_dir}")
        if not self.files:
            raise FileNotFoundError(f"Aucun fichier .pt trouvé dans {preprocessed_dir}." \
                  " Avez-vous exécuté le script de pré-calcul ?")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Le chargement d'un fichier .pt est EXTRÊMEMENT rapide
        return torch.load(self.files[idx])

# --- Collate_fn (toujours nécessaire) ---
def collate_fn(batch):
    """Filtre les échantillons 'None' (au cas où, bien que non nécessaire ici)"""
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return tuple(zip(*batch))

# ----------------------------
# 5. Initialisation Dataset et DataLoader (OPTIMISÉ)
# ----------------------------

# Utilise le nouveau Dataset pointant vers les fichiers .pt
dataset = PreprocessedSARDDataset(PREPROCESSED_DIR_TRAIN)

data_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,         # 🚀 L'accélération vient d'ici
    pin_memory=True,                 # 🚀 Et d'ici
)

print(f" Dataset: {len(dataset)} images pré-calculées")
print(f" Batch size: {BATCH_SIZE}")
print(f" Workers: {NUM_WORKERS}")


# ----------------------------
# 6. Définition du modèle Mask R-CNN (inchangé)
# ----------------------------
print("\n Initialisation du modèle Mask R-CNN...")
num_classes = NUM_CLASSES_MASKRCNN
model = maskrcnn_resnet50_fpn(weights=True)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
model.to(DEVICE)

# Mode d'entraînement en précision mixte (FP16)
scaler = torch.amp.GradScaler('cuda')

# ----------------------------
# 7. Optimiseur (inchangé)
# ----------------------------
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=0.0003, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# ----------------------------
# 8. Boucle d'Entraînement (presque inchangée)
# ----------------------------
num_epochs = NUM_EPOCHS_MASKRCNN
print(f"\n Démarrage de l'entraînement optimisé pour {num_epochs} époques\n")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    batch_count = 0

    # Le 'data_loader' est maintenant beaucoup plus rapide
    pbar = tqdm(data_loader,
                desc=f"Epoch {epoch+1}/{num_epochs}",
                ncols=100,
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')
    start_time = time.time()

    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        # Le reste de la boucle est identique à votre code

        torch.cuda.empty_cache()

        try:
            imgs, targets = batch
            imgs = list(img.to(DEVICE) for img in imgs)
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            with torch.amp.autocast('cuda'):
                loss_dict = model(imgs, targets)
                losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(losses).backward()
            scaler.step(optimizer)
            scaler.update()

            del imgs, targets, loss_dict
            torch.cuda.empty_cache()

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"WARNING: 'out of memory' error on batch {batch_idx}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            else:
                raise e

        epoch_loss += losses.item()
        batch_count += 1
        avg_loss = epoch_loss / batch_count
        pbar.set_postfix({'Loss': f'{avg_loss:.4f}', 'Batch Loss': f'{losses.item():.4f}'})

    epoch_time = time.time() - start_time
    avg_epoch_loss = epoch_loss / batch_count if batch_count > 0 else 0
    lr_scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    print(f" Epoch {epoch+1}/{num_epochs} finished | "
          f"Loss: {avg_epoch_loss:.4f} | "
          f"Time: {epoch_time:.1f}s | "
          f"LR: {current_lr:.6f}")

    # Sauvegarde à chaque époque (recommandé pour les longs entraînements)
    epoch_save_path = MASK_RCNN_PATH.replace(".pth", f"_epoch{epoch+1}.pth")
    torch.save(model.state_dict(), epoch_save_path)
    print(f" Checkpoint de l'époque {epoch+1} sauvegardé : {epoch_save_path}")


# ----------------------------
# 9. Sauvegarde du modèle final (corrigé)
# ----------------------------
torch.save(model.state_dict(), MASK_RCNN_PATH) # Utilise la variable correcte
print("\n Modèle entraîné et sauvegardé avec succès!")
print(f" Fichier final: {MASK_RCNN_PATH}")

Trouvé 2107 fichiers pré-calculés dans ./dataset_preprocessed/train
 Dataset: 2107 images pré-calculées
 Batch size: 4
 Workers: 0

 Initialisation du modèle Mask R-CNN...


c:\Users\ac223162\AppData\Local\anaconda3\envs\mlenv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



 Démarrage de l'entraînement optimisé pour 2 époques



Epoch 1/2:   0%|                                                            | 0/527 [00:00<?, ?it/s]C:\Users\ac223162\AppData\Local\Temp\ipykernel_23864\253594195.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issu

# Step 4: Prediction and Visualization

Using the trained models (YOLO and Mask R-CNN) to make predictions on a test image.

In [25]:
def detect_image_yolo(img_path):

    # Charger le modèle
    try:
        # Assurez-vous que YOLO_BEST_PT est défini globalement
        model = YOLO(YOLO_BEST_PT)
    except NameError:
        print("Erreur: 'YOLO_BEST_PT' n'est pas défini.")
        # Tenter de retourner l'image originale si le modèle échoue
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
             raise FileNotFoundError(f"Could not read image: {img_path}")
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Faire l'inférence
    results = model(img_path, conf=0.25)

    # --- MODIFICATION : DESSIN MANUEL STYLE MASK R-CNN ---

    # 1. Lire l'image originale
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")
    # Convertir en RGB pour le dessin et l'affichage
    annotated_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Essayer d'obtenir la map de classes
    # (Pour la cohérence des noms avec Mask R-CNN si 'classes' est défini)
    try:
        classes_map_yolo = classes
    except NameError:
        # Sinon, utiliser les noms du modèle YOLO
        classes_map_yolo = results[0].names
        if classes_map_yolo is None:
            print("Warning: 'classes' map not found. Using default names.")
            classes_map_yolo = {0: "class_0"}

    # 3. Itérer sur les détections
    detections = results[0].boxes

    for i, detection in enumerate(detections):
        # Extraire les informations de la détection
        box = detection.xyxy[0].cpu().numpy().astype(int) # Coordonnées [xmin, ymin, xmax, ymax]
        xmin, ymin, xmax, ymax = box
        score = detection.conf[0].cpu().numpy()
        class_idx = int(detection.cls[0].cpu().numpy()) # Index de classe

        # Obtenir le nom de la classe
        class_name = classes_map_yolo.get(class_idx, str(class_idx))

        # --- Logique de dessin copiée de Mask R-CNN ---

        # 4. Couleur aléatoire mais stable (basée sur l'index de classe et l'instance)
        rnd = (class_idx * 37 + i * 17) % 255
        color = (int((rnd * 97) % 255), int((rnd * 67) % 255), int((rnd * 137) % 255))
        # Note: 'color' est en RGB car 'annotated_img' est en RGB

        # 5. Dessiner la boîte (rectangle)
        cv2.rectangle(annotated_img, (xmin, ymin), (xmax, ymax), color, 2)

        # --- AMÉLIORATION DE LISIBILITÉ (Style YOLO) ---

        # 6. Définir le texte (label + score)
        txt = f"{class_name} {score:.2f}"

        # 7. Obtenir la taille du texte
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5
        font_thickness = 1
        (w, h), baseline = cv2.getTextSize(txt, font, font_scale, font_thickness)

        # 8. S'assurer que le texte ne sort pas en haut
        t_ymin = max(ymin - h - 6, 0)

        # 9. Dessiner le rectangle de fond rempli
        cv2.rectangle(annotated_img,
                      (xmin, t_ymin),
                      (xmin + w, t_ymin + h + baseline + 3),
                      color,
                      -1) # -1 pour remplir

        # 10. Dessiner le texte (en blanc pour contraster)
        cv2.putText(annotated_img,
                    txt,
                    (xmin, t_ymin + h + 1), # Positionner le texte dans le fond
                    font,
                    font_scale,
                    (255, 255, 255), # Couleur du texte (blanc)
                    font_thickness,
                    cv2.LINE_AA)

    # --- Fin de la modification ---

    # Sauvegarder l'image (convertir RGB -> BGR pour cv2.imwrite)
    cv2.imwrite("annotated_test.jpg", cv2.cvtColor(annotated_img, cv2.COLOR_RGB2BGR))
    print("Image annotée sauvegardée -> annotated_test.jpg")

    # Retourner l'image annotée (format RGB)
    return annotated_img

In [26]:

def detect_image_maskrcnn(img_path):
    checkpoint_path = "mask_rcnn_sard2_yolo_final.pth"

    # --- Load checkpoint and infer num_classes from it (preferred) ---
    if os.path.exists(checkpoint_path):
        state = torch.load(checkpoint_path, map_location='cpu')
        sd = state['model_state_dict'] if isinstance(state, dict) and 'model_state_dict' in state else state

        # Try to find the cls_score weight key to infer number of classes
        cls_key = None
        for k in sd.keys():
            if k.endswith('roi_heads.box_predictor.cls_score.weight') or 'cls_score.weight' in k:
                cls_key = k
                break

        if cls_key is not None:
            nc = sd[cls_key].shape[0]
            print(f"Inferred num_classes from checkpoint: {nc}")
        else:
            # Fallback to global / configured value
            try:
                nc = num_classes
            except NameError:
                try:
                    nc = NUM_CLASSES_MASKRCNN
                except NameError:
                    nc = 2
            print(f"Could not infer num_classes from checkpoint, using nc={nc}")

        # Build model with inferred num_classes
        model = maskrcnn_resnet50_fpn(weights=None, num_classes=nc)

        # Load state dict, try strict=True then strict=False if mismatch
        try:
            model.load_state_dict(sd)
            print("Checkpoint loaded with strict=True")
        except Exception as e:
            print("Strict load failed (likely head size mismatch). Loading with strict=False and keeping unmatched heads reinitialized.")
            missing = model.load_state_dict(sd, strict=False)
            print("Missing keys / unexpected keys:", missing)
    else:
        print(f"Checkpoint not found: {checkpoint_path}. Building model with default num_classes.")
        try:
            nc = num_classes
        except NameError:
            nc = NUM_CLASSES_MASKRCNN if 'NUM_CLASSES_MASKRCNN' in globals() else 2
        model = maskrcnn_resnet50_fpn(weights=None, num_classes=nc)

    # Ensure device variable exists
    device_to_use = DEVICE if 'DEVICE' in globals() else torch.device('cpu')
    model.to(device_to_use)
    model.eval()

    # Use the CLASSES mapping defined in cell 2 if available
    if 'CLASSES' in globals():
        # CLASSES keys are original class ids (0..N-1)
        classes_map = CLASSES
    else:
        # fallback: build mapping from inferred nc (nc includes background)
        original_classes = max(0, nc - 1)
        classes_map = {i: f"class_{i}" for i in range(original_classes)}

    # Read and prepare image for Mask R-CNN
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]

    # Convert to tensor expected by torchvision
    img_tensor = torchvision.transforms.functional.to_tensor(img_rgb).to(device_to_use)

    with torch.no_grad():
        outputs = model([img_tensor])

    out = outputs[0]
    scores = out.get('scores', torch.tensor([])).cpu()
    boxes = out.get('boxes', torch.tensor([])).cpu()
    labels = out.get('labels', torch.tensor([])).cpu()
    masks = out.get('masks', None)
    if masks is not None:
        masks = masks.cpu()

    # Filter by score threshold
    score_thresh = 0.5
    keep = (scores >= score_thresh).nonzero(as_tuple=False).squeeze(1)

    annotated = img_rgb.copy()

    if keep.numel() == 0:
        print("No detections above threshold")
    else:
        boxes = boxes[keep].numpy().astype(int)
        scores = scores[keep].numpy()
        labels = labels[keep].numpy()
        if masks is not None:
            masks = masks[keep].numpy()  # shape [N,1,H,W]

        alpha = 0.5
        for i, box in enumerate(boxes):
            xmin, ymin, xmax, ymax = box
            score = scores[i]
            label = int(labels[i])

            # Mask R-CNN labels include background at 0, training used +1 for real classes
            class_idx = label - 1
            # Safe lookup in CLASSES mapping
            class_name = classes_map.get(class_idx, f"class_{class_idx}")

            rnd = (class_idx * 37 + i * 17) % 255
            color = (int((rnd * 97) % 255), int((rnd * 67) % 255), int((rnd * 137) % 255))

            # Draw box and label (YOLO-style background)
            cv2.rectangle(annotated, (xmin, ymin), (xmax, ymax), color, 2)
            txt = f"{class_name} {score:.2f}"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.5
            font_thickness = 1
            (w, h), baseline = cv2.getTextSize(txt, font, font_scale, font_thickness)
            t_ymin = max(ymin - h - 6, 0)
            cv2.rectangle(annotated,
                          (xmin, t_ymin),
                          (xmin + w, t_ymin + h + baseline + 3),
                          color,
                          -1)
            cv2.putText(annotated,
                        txt,
                        (xmin, t_ymin + h + 1),
                        font,
                        font_scale,
                        (255, 255, 255),
                        font_thickness,
                        cv2.LINE_AA)

            # Draw mask if present
            if masks is not None:
                mask = masks[i][0]  # [H,W]
                mask_bin = (mask >= 0.3).astype('uint8')
                if mask_bin.sum() > 0:
                    colored_mask = np.zeros_like(annotated, dtype=np.uint8)
                    colored_mask[:, :] = color
                    annotated = np.where(mask_bin[:, :, None].astype(bool),
                                         (annotated * (1 - alpha) + colored_mask * alpha).astype(np.uint8),
                                         annotated)

    # Save Mask R-CNN result
    out_bgr = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
    out_fname = "annotated_maskrcnn.jpg"
    cv2.imwrite(out_fname, out_bgr)
    print(f"Annotated image saved -> {out_fname}")

    return annotated
# ...existing code...

In [29]:
# select a random test image
img_paths = glob.glob("./dataset/images/test/*.jpg")
img_path = random.choice(img_paths)
print(f"Image test: {img_path}")
# Run YOLO detection
annotated_img=detect_image_yolo(img_path=img_path)
# Run Mask R-CNN detection
annotated=detect_image_maskrcnn(img_path=img_path)

plt.figure(figsize=(20,8))

plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(cv2.imread("annotated_test.jpg"), cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("YOLO Detections")

plt.subplot(1,2,2)
plt.imshow(annotated)
plt.axis('off')
plt.title("Mask R-CNN Detections")

plt.tight_layout()
plt.show()

Image test: ./dataset/images/test\gss531_jpg.rf.9e7caa68afd971068da145ab4cd4a629.jpg

image 1/1 c:\Users\ac223162\Desktop\esilv\annee_4\machine_learning\Projet\dataset\images\test\gss531_jpg.rf.9e7caa68afd971068da145ab4cd4a629.jpg: 384x640 1 Walking, 43.4ms
Speed: 3.9ms preprocess, 43.4ms inference, 12.2ms postprocess per image at shape (1, 3, 384, 640)
Image annotée sauvegardée -> annotated_test.jpg
Inferred num_classes from checkpoint: 7
Checkpoint loaded with strict=True
No detections above threshold
Annotated image saved -> annotated_maskrcnn.jpg


<Figure size 2000x800 with 2 Axes>

In [ ]:
# ===================================================================
# ÉTAPE 5.1 : ÉVALUATION QUANTITATIVE (BASELINE YOLO)
# ===================================================================
# (Cette cellule est NOUVELLE et répond aux exigences du PDF)

print("--- ÉVALUATION YOLO (BASELINE) SUR LE JEU DE TEST ---")

# Assurez-vous d'avoir le bon chemin vers votre meilleur modèle
# (Adaptez le chemin si 'sard2_yolo11_augmented7' n'est pas le bon nom)
try:
    model_yolo_best = YOLO(YOLO_BEST_PT)

    # Lancer l'évaluation sur le jeu de test (défini dans data.yaml)
    metrics = model_yolo_best.val(split='test', data=DATA_YAML)

    print("\n--- RÉSULTATS YOLO ---")
    print(f"mAP@50-95 : {metrics.box.map:.4f}")
    print(f"mAP@50    : {metrics.box.map50:.4f}")
    print(f"Précision : {metrics.box.p[0]:.4f}") # Précision pour la 1ère classe
    print(f"Rappel    : {metrics.box.r[0]:.4f}") # Rappel pour la 1ère classe

except Exception as e:
    print(f"Erreur lors de l'évaluation YOLO: {e}")
    print(f"Vérifiez que le chemin '{YOLO_BEST_PT}' est correct.")

--- ÉVALUATION YOLO (BASELINE) SUR LE JEU DE TEST ---
Ultralytics 8.3.209  Python-3.11.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 228.458.6 MB/s, size: 993.0 KB)
val: Scanning C:\Users\ac223162\Desktop\esilv\annee_4\machine_learning\Projet\dataset_augmente\labels\test... 387 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 387/387 893.2it/s 0.4s0.0s
val: New cache created: C:\Users\ac223162\Desktop\esilv\annee_4\machine_learning\Projet\dataset_augmente\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 5.5it/s 4.6s0.1s
                   all        387       1228      0.795      0.499      0.574       0.31
               Running         11         15          1          0     0.0689     0.0254
               Walking        151        242      0

In [ ]:
# ===================================================================
# ÉTAPE 5.2 : ÉVALUATION QUANTITATIVE (AVANCÉ MASK R-CNN)
# ===================================================================
# (Cette cellule est NOUVELLE et répond aux exigences du PDF)

print("--- ÉVALUATION MASK R-CNN (AVANCÉ) SUR LE JEU DE TEST ---")

# Import nécessaire pour les métriques
try:
    from torchmetrics.detection import MeanAveragePrecision
    TORCHMETRICS_AVAILABLE = True
except ImportError:
    print("ERREUR: 'torchmetrics' n'est pas installé. "
          "Veuillez l'installer : pip install torchmetrics")
    TORCHMETRICS_AVAILABLE = False

if TORCHMETRICS_AVAILABLE:

    # 1. Préparer le DataLoader de TEST
    # (Utilise votre SARD2YOLODataset et vos variables globales)
    try:
        test_dataset = SARD2YOLODataset(AUG_IMG_SOURCE_TEST, AUG_LBL_SOURCE_TEST)
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE_MASKRCNN, # Utilise votre BATCH_SIZE global
            shuffle=False,
            collate_fn=collate_fn, # Utilise votre collate_fn
            num_workers=NUM_WORKERS
        )
        print(f"Jeu de test chargé : {len(test_dataset)} images.")
    except Exception as e:
        print(f"Erreur lors du chargement du jeu de test: {e}")
        print("Vérifiez vos variables AUG_IMG_SOURCE_TEST/AUG_LBL_SOURCE_TEST.")

    # 2. Charger le modèle sauvegardé
    try:
        # Re-créer l'architecture du modèle
        model_eval = maskrcnn_resnet50_fpn(weights=None, num_classes=NUM_CLASSES_MASKRCNN)

        # Charger les poids sauvegardés
        model_eval.load_state_dict(torch.load(MASK_RCNN_PATH, map_location=DEVICE))
        model_eval.to(DEVICE)
        model_eval.eval() # Mettre en mode évaluation
        print(f"Modèle chargé depuis : {MASK_RCNN_PATH}")
    except Exception as e:
        print(f"Erreur lors du chargement du modèle Mask R-CNN: {e}")
        print("Assurez-vous que le modèle s'est bien entraîné et sauvegardé.")

    # 3. Initialiser l'outil de métrique
    metric = MeanAveragePrecision(iou_type="bbox").to(DEVICE)

    # 4. Boucle d'évaluation
    print("Démarrage de l'évaluation sur le jeu de test...")
    with torch.no_grad(): # Pas besoin de calculer les gradients
        for batch in tqdm(test_loader, desc="Évaluation Mask R-CNN"):
            if batch is None:
                continue

            images, targets = batch
            images = list(img.to(DEVICE) for img in images)
            targets_on_device = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            # Obtenir les prédictions
            predictions = model_eval(images)

            # Mettre à jour les métriques
            # Ajustement : les prédictions (preds) ont des labels 1-N (fond=0)
            # Les cibles (targets) ont des labels 0-(N-1)
            # Nous devons aligner les cibles sur les prédictions (+1)
            formatted_targets = []
            for t in targets_on_device:
                formatted_targets.append({
                    "boxes": t["boxes"],
                    # +1 pour aligner sur les labels du modèle (1-6)
                    "labels": t["labels"] + 1
                })

            metric.update(predictions, formatted_targets)

    # 5. Calculer et afficher les résultats finaux
    try:
        results = metric.compute()
        print("\n--- RÉSULTATS MASK R-CNN ---")
        print(f"mAP@50-95 : {results['map'].item():.4f}")
        print(f"mAP@50    : {results['map_50'].item():.4f}")
        print(f"Recall (large): {results['recall_large'].item():.4f}")
    except Exception as e:
        print(f"Erreur lors du calcul final des métriques : {e}")

--- ÉVALUATION MASK R-CNN (AVANCÉ) SUR LE JEU DE TEST ---
Jeu de test chargé : 387 images.
Modèle chargé depuis : mask_rcnn_sard2_yolo_final.pth
Démarrage de l'évaluation sur le jeu de test...


Évaluation Mask R-CNN: 100%|██████████| 194/194 [10:36<00:00,  3.28s/it]



--- RÉSULTATS MASK R-CNN ---
mAP@50-95 : 0.0223
mAP@50    : 0.0594
Erreur lors du calcul final des métriques : 'recall_large'
